# Chapter 10 &mdash; The Matching Algorithm: Steer by Derivatives, Decide by Nullability

**Concept 3 of the Chapter 10 decomposition:** *The Matching Algorithm: Steer by Derivatives, Decide by Nullability*

Consume $w$ one symbol at a time taking derivatives; accept iff the final expression is nullable.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter10/Concept-Matching-Algorithm/Concept-Matching-Algorithm.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_rederiv    import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The algorithm in two lines:

```
for c in w:  E = dv(c, E)
return nullable(E)
```

**Derivatives steer; nullability decides.** Every symbol rewrites the expression, and
only at the end do you ask whether what remains can match $\varepsilon$.

The correctness argument is a one-line induction on $|w|$: $w \in L(E)$ iff
$w' \in L(E_c)$ where $w = cw'$, with $\varepsilon \in L(E)$ iff $E$ is nullable as the
basis.

Cost: one rewrite per symbol. Without simplification the expression can grow, which
Concept 8 addresses with smart constructors.

## 2. Definitions

### The matcher

In [ ]:
# --- the derivative matcher, in full -------------------------------------
# AST forms produced by re2ast:
#    ('@','@')            epsilon
#    ('str', c)           a single symbol
#    ('+', (E1, E2))      union
#    ('.', (E1, E2))      concatenation
#    ('*', E)             star
#    ('!', E)             negation
#    ('&', (E1, E2))      intersection
EPS   = ('@', '@')
PHI   = ('phi', 'phi')          # the empty language -- not produced by the
                                # parser, but the derivative rules need it

def nullable(E):
    t = E[0]
    if t == '@'  : return True
    if t == 'phi': return False
    if t == 'str': return False
    if t == '+'  : return nullable(E[1][0]) or  nullable(E[1][1])
    if t == '&'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '.'  : return nullable(E[1][0]) and nullable(E[1][1])
    if t == '*'  : return True
    if t == '!'  : return not nullable(E[1])
    raise ValueError(E)

def dv(c, E):
    t = E[0]
    if t == '@'  : return PHI
    if t == 'phi': return PHI
    if t == 'str': return EPS if E[1] == c else PHI
    if t == '+'  : return ('+', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '&'  : return ('&', (dv(c, E[1][0]), dv(c, E[1][1])))
    if t == '*'  : return ('.', (dv(c, E[1]), E))
    if t == '!'  : return ('!', dv(c, E[1]))
    if t == '.'  :
        E1, E2 = E[1]
        left = ('.', (dv(c, E1), E2))
        return ('+', (left, dv(c, E2))) if nullable(E1) else left
    raise ValueError(E)

def matches(s, E):
    for ch in s:
        E = dv(ch, E)
    return nullable(E)

def rmatch(restr, s):
    return matches(s, re2ast(restr)[0])

### An instrumented version that shows the two phases

In [ ]:
def match_verbose(restr, s):
    E = re2ast(restr)[0]
    for i, ch in enumerate(s, 1):
        E = dv(ch, E)
        print("  step %d: consumed '%s', nullable so far = %s" % (i, ch, nullable(E)))
    print("  decide : nullable(final) = %s" % nullable(E))
    return nullable(E)

## 3. Tests

Steer, then decide.

In [ ]:
print("matching '0101' against (0+1)*01")
r = match_verbose("(0+1)*01", "0101")
assert r
print()
print("matching '0110' against (0+1)*01")
r = match_verbose("(0+1)*01", "0110")
assert not r

The basis case: an empty input decides on nullability alone.

In [ ]:
for restr in ["0*", "0+1", "''", "(0+1)*01"]:
    E = re2ast(restr)[0]
    print("%-12s nullable? %-6s matches ''? %s" % (restr, nullable(E), matches('', E)))
    assert nullable(E) == matches('', E)

The induction, checked directly: $w\in L(E) \iff w' \in L(E_c)$.

In [ ]:
from itertools import product
E = re2ast("(0+1)*01")[0]
strs = [''.join(p) for k in range(1, 9) for p in product('01', repeat=k)]
assert all(matches(w, E) == matches(w[1:], dv(w[0], E)) for w in strs)
print("peeling one symbol agrees with taking one derivative, on all %d strings"
      % len(strs))

And the whole matcher agrees with the DFA route.

In [ ]:
for restr in ["0*1", "(0+1)*01", "(01)*", "0*1*", "(0+1)*1(0+1)(0+1)"]:
    D = min_dfa(nfa2dfa(re2nfa(restr)))
    strs = [''.join(p) for k in range(10) for p in product('01', repeat=k)]
    bad = [s for s in strs if rmatch(restr, s) != accepts_dfa(D, s)]
    print("%-22s mismatches: %d" % (restr, len(bad)))
    assert not bad

Cost is one rewrite per symbol &mdash; but the expression can grow.

In [ ]:
E = re2ast("(0+1)*01")[0]
def size(E):
    if E[0] in ('@', 'phi', 'str'): return 1
    if E[0] in ('*', '!'): return 1 + size(E[1])
    return 1 + size(E[1][0]) + size(E[1][1])
for i, ch in enumerate('010101', 1):
    E = dv(ch, E)
    print("  after %d symbols, AST size %d" % (i, size(E)))
print("\nUnsimplified growth is the problem Concept 8 solves.")

## 4. Exercises


1. Write the correctness induction out in full.
2. What is the worst-case AST growth over $n$ symbols, unsimplified?
3. Instrument `matches` to report the largest AST it ever holds.

In [ ]:
# Your work for the exercises above.